# 📶 Starlink Ku/Ka-Band RF Link Budget & DVB-S2X Waterfall Analysis
## Interactive Engineering Notebook: FSPL, Rain Fade & Carrier-to-Noise Ratio

> **Lead Disciplines:** `telecom-mathematics-expert` & `radio-frequency-specialist`  
> **Standards Reference:** StarSec CSF v1.0, Technical Monograph Chapter 06 & R&D Whitepaper 02

This notebook provides interactive decibel telecommunications link budget modeling for Starlink Ku-band downlink (12 GHz, 250 MHz channel) and Ka/V-band gateway feeder links. It computes Free-Space Path Loss (FSPL), atmospheric and ITU-R P.618 rain attenuation, Receiver G/T, and visualizes cascaded power waterfall budgets against DVB-S2X MODCOD thresholds.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Ensure local simulations module is importable
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root))

from simulations.rf_link_budget_calculator import (
    LinkParameters,
    LinkBudgetResult,
    calculate_link_budget,
    calculate_slant_range_km
)

print("✅ Successfully imported RF Link Budget Simulation Engine")

### 1. Ku-Band Downlink Nominal Parameter Setup

Spacecraft transmitter power is $\approx 20\text{ W}$ ($+13.0\text{ dBW}$) feeding a phased array with $+38.0\text{ dBi}$ gain, producing an EIRP of $+50.0\text{ dBW}$. The User Terminal (Dishy) provides $+34.0\text{ dBi}$ gain with an LNA noise figure of $1.5\text{ dB}$.

In [ ]:
ku_downlink_clear = LinkParameters(
    link_name="Starlink Ku-Band Downlink (Clear Sky)",
    carrier_frequency_ghz=12.0,
    bandwidth_mhz=250.0,
    data_rate_mbps=220.0,
    slant_range_km=calculate_slant_range_km(altitude_km=550.0, elevation_deg=45.0),
    tx_power_watts=20.0,
    tx_antenna_gain_dbi=38.0,
    tx_cable_loss_db=1.0,
    tx_pointing_loss_db=0.5,
    rx_antenna_gain_dbi=34.0,
    rx_pointing_loss_db=0.5,
    rx_noise_figure_db=1.5,
    antenna_noise_temp_k=60.0,
    atmospheric_loss_db=0.4,
    rain_attenuation_db=0.0,
    required_eb_n0_db=7.5    # 16APSK 3/4 DVB-S2X threshold
)

res_clear = calculate_link_budget(ku_downlink_clear)

print(f"Link Budget Execution Results (Elevation = 45°):")
print(f"  • Slant Range             : {ku_downlink_clear.slant_range_km:.1f} km")
print(f"  • Free-Space Path Loss    : {res_clear.free_space_path_loss_db:.2f} dB")
print(f"  • Spacecraft EIRP         : {res_clear.eirp_dbw:.2f} dBW")
print(f"  • Receiver G/T            : {res_clear.receiver_g_over_t_db_k:.2f} dB/K")
print(f"  • Received C/N0           : {res_clear.c_over_n0_db_hz:.2f} dB-Hz")
print(f"  • Achieved Eb/N0          : {res_clear.eb_n0_db:.2f} dB")
print(f"  • Link Margin             : +{res_clear.margin_db:.2f} dB ({'CLOSES' if res_clear.link_closes else 'FAILS'})")

### 2. Cascaded Decibel Power Level Waterfall Chart

The waterfall chart tracks signal progression from the spacecraft traveling-wave tube amplifier (TWTA), through free space and atmosphere, into Dishy baseband processing.

In [ ]:
stages = ["Tx Power", "Tx Gain", "Tx Losses", "FSPL", "Atmosphere", "Rx Gain", "Rx Losses"]
deltas = [
    10 * np.log10(ku_downlink_clear.tx_power_watts),  # +13 dBW
    ku_downlink_clear.tx_antenna_gain_dbi,            # +38 dB
    -(ku_downlink_clear.tx_cable_loss_db + ku_downlink_clear.tx_pointing_loss_db), # -1.5 dB
    -res_clear.free_space_path_loss_db,               # ~ -171.5 dB
    -ku_downlink_clear.atmospheric_loss_db,           # -0.4 dB
    ku_downlink_clear.rx_antenna_gain_dbi,            # +34 dB
    -ku_downlink_clear.rx_pointing_loss_db            # -0.5 dB
]

cumulative = np.cumsum(deltas)

plt.figure(figsize=(10, 4.5))
bars = plt.bar(stages, deltas, color=['g', 'g', 'r', 'r', 'r', 'g', 'r'], alpha=0.7)
plt.plot(stages, cumulative, 'k-o', lw=2, label="Cumulative Power (dBW)")
plt.title("Starlink Ku-Band Downlink Cascaded Power Waterfall")
plt.ylabel("Decibels (dB / dBW)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### 3. Elevation Angle vs Link Margin & Rain Fade Resilience

As satellites track across the sky from horizon ($25^\circ$ minimum mask) to zenith ($90^\circ$), path distance decreases from $\approx 1,120\text{ km}$ to $550\text{ km}$, yielding $\approx 6.2\text{ dB}$ higher received carrier power.

In [ ]:
elevations = np.linspace(25, 90, 66)
margins_clear = []
margins_rain = []

for el in elevations:
    slant = calculate_slant_range_km(550.0, el)
    # Clear sky
    p_c = LinkParameters(**{**ku_downlink_clear.__dict__, 'slant_range_km': slant, 'rain_attenuation_db': 0.0})
    margins_clear.append(calculate_link_budget(p_c).margin_db)
    # Heavy rain (3.5 dB fade at 12 GHz)
    p_r = LinkParameters(**{**ku_downlink_clear.__dict__, 'slant_range_km': slant, 'rain_attenuation_db': 3.5})
    margins_rain.append(calculate_link_budget(p_r).margin_db)

plt.figure(figsize=(9, 4.5))
plt.plot(elevations, margins_clear, 'b-', lw=2, label="Clear Sky Availability")
plt.plot(elevations, margins_rain, 'r--', lw=2, label="Heavy Rain Fade (-3.5 dB)")
plt.axhline(0, color='black', lw=1, linestyle=':', label="Threshold Limit (0 dB)")
plt.title("Downlink Operating Margin vs Satellite Elevation Angle")
plt.xlabel("Satellite Elevation Angle (degrees)")
plt.ylabel("Link Margin (dB)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()